# 第15章 次同步谐振 (Subsynchronous Resonance)

> Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 15

## 本章概述

次同步谐振 (SSR) 是串联补偿输电线路与汽轮发电机轴系之间
的机电耦合振荡现象，频率低于同步频率 (50/60 Hz)。

### SSR 的三种形式
1. **感应发电机效应** (IGE): 次同步频率下发电机呈现负电阻
2. **扭振相互作用** (TI): 电气谐振与轴系扭振模式的耦合
3. **暂态扭矩放大** (TTA): 故障后的大幅扭振

### 电气谐振频率
$$f_{er} = f_0 \\sqrt{\\frac{X_C}{X_d'' + X_T + X_L}}$$
当 $f_{er}$ 接近轴系扭振频率的互补频率 ($f_0 - f_m$) 时，
可能发生扭振相互作用。

> 注: 本章为简化处理，完整的 SSR 分析需要详细的轴系多质量模型
> 和电磁暂态仿真，属于研究生级别内容。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('Import OK')

---
## 15.1 电气谐振频率

串联补偿线路形成 RLC 谐振回路，谐振频率取决于补偿度和系统参数。

### 补偿度
$$K = \\frac{X_C}{X_L} \\times 100\\%$$

典型补偿度: 30-70%，对应的电气谐振频率: 25-45 Hz (50 Hz 系统)。

In [ ]:
# ===== 电气谐振频率 vs 补偿度 =====
f0 = 50  # Hz
K_vals = np.linspace(10, 90, 81)  # compensation level %
fer = f0 * np.sqrt(K_vals / 100)  # electrical resonance freq

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_vals, fer, 'b-', linewidth=2)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Sync freq (50 Hz)')
ax.fill_between(K_vals, 0, 50, alpha=0.1, color='red', label='Subsynchronous region')
ax.set_xlabel('Compensation Level K (%)')
ax.set_ylabel('Electrical Resonance Frequency (Hz)')
ax.set_title('Electrical Resonance Frequency vs Series Compensation')
ax.grid(True, alpha=0.3); ax.legend()

# Typical compensation
for Ktyp in [30, 50, 70]:
    ftyp = f0 * np.sqrt(Ktyp / 100)
    ax.plot([Ktyp], [ftyp], 'ro', markersize=6)
    ax.annotate(f'K={Ktyp}%: f={ftyp:.1f} Hz', (Ktyp, ftyp),
               textcoords='offset points', xytext=(10, -15), fontsize=9)
plt.tight_layout()
plt.savefig('examples/kundur/ssr_resonance_freq.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'K=30%: f_er={f0*np.sqrt(0.3):.1f} Hz')
print(f'K=50%: f_er={f0*np.sqrt(0.5):.1f} Hz')
print(f'K=70%: f_er={f0*np.sqrt(0.7):.1f} Hz')

---
## 15.2 次同步频率下的发电机阻抗

在次同步频率 $f_m$ 下，同步发电机呈现负电阻特性（感应发电机效应）。

发电机等效阻抗 (次同步频率 $f_m$):
$$Z_G(f_m) = R_a + j\\left(\\frac{f_m}{f_0}\\right) X_d''$$

但电枢旋转产生的感应效应使有效电阻为:
$$R_{eff} \\approx \\frac{R_r (1-s)}{s}$$
其中 $s = (f_0 - f_m)/f_0$ 为转差率。

当 $f_m < f_0$ (次同步) 时，$s > 0$，$R_{eff} < 0$ → **负电阻** → 电气振荡发散！

In [ ]:
# ===== 感应发电机效应: 负电阻 vs 频率 =====
fm = np.linspace(5, 45, 200)  # subsynchronous freq
s = (f0 - fm) / f0  # slip
Rr = 0.02  # rotor resistance (pu)
Reff = Rr * (1 - s) / np.where(np.abs(s) > 1e-6, s, 1e-6)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fm, Reff, 'r-', linewidth=2)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.fill_between(fm, Reff, 0, where=(Reff<0), alpha=0.2, color='red', label='Negative resistance')
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5, label='Sync speed')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Effective Rotor Resistance (pu)')
ax.set_title('Induction Generator Effect: Negative Resistance at Subsynchronous Freq')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/ssr_negative_resistance.png', dpi=150, bbox_inches='tight')
plt.show()
print('At subsynchronous frequencies: Reff < 0 -> negative damping')
print('This is the fundamental cause of SSR induction generator effect')

---
## 15.3 简化的轴系扭振模型

大型汽轮发电机轴系包含多个质量块 (HP, IP, LP, GEN, EXC)，
形成多个扭振模式。

### 两质量简化模型
$$\\begin{bmatrix} J_1 & 0 \\\\ 0 & J_2 \\end{bmatrix}
\\begin{bmatrix} \\ddot{\\theta}_1 \\\\ \\ddot{\\theta}_2 \\end{bmatrix} +
\\begin{bmatrix} K & -K \\\\ -K & K \\end{bmatrix}
\\begin{bmatrix} \\theta_1 \\\\ \\theta_2 \\end{bmatrix} = 0$$

特征频率: $f_t = \\frac{1}{2\\pi}\\sqrt{K\\frac{J_1+J_2}{J_1 J_2}}$

In [ ]:
# ===== 两质量轴系扭振频率 =====
# Turbine + Generator (simplified)
J_gen = 5000  # kg-m^2 (generator rotor)
J_turb = 3000  # kg-m^2 (turbine equivalent)
K_shaft = 80e6  # Nm/rad (shaft stiffness)

omega_t = np.sqrt(K_shaft * (J_gen + J_turb) / (J_gen * J_turb))
f_torsional = omega_t / (2 * np.pi)

print(f'Torsional mode frequency: {f_torsional:.2f} Hz')
print(f'Electrical resonance K=30%: {f0*np.sqrt(0.3):.1f} Hz')
print(f'Complementary freq (50Hz - ft): {50 - f_torsional:.2f} Hz')

# Typical multi-mass torsional frequencies for large steam turbines:
torsional_modes = {
    'Mode 0 (rigid body)': 0.0,
    'Mode 1': 15.7,
    'Mode 2': 20.2,
    'Mode 3': 25.5,
    'Mode 4': 32.3,
    'Mode 5': 47.5,
}

print('\nTypical torsional modes (IEEE SSR Benchmark):')
for name, f_t in torsional_modes.items():
    f_comp = 50 - f_t  # complementary (electrical) frequency
    print(f'  {name}: {f_t:.1f} Hz -> complementary: {f_comp:.1f} Hz')
    if 15 < f_comp < 45:
        print(f'    *** RISK: f_comp in typical compensation range ***')

---
## 15.4 频率扫描分析

从发电机转子侧看入系统的等效阻抗随频率的变化。
当电抗为零且电阻为负时，存在 SSR 风险。

In [ ]:
# ===== 频率扫描: 系统阻抗 vs 频率 =====
f_scan = np.linspace(5, 55, 500)
omega_scan = 2 * np.pi * f_scan

# Simplified RLC model of series-compensated line
L_line = 0.5 / (2 * np.pi * 50)  # X=0.5 pu -> L
R_line = 0.02
C_comp = 1.0 / (0.5 * 0.5 * (2*np.pi*50)**2 * L_line)  # 50% compensation

Z_line = R_line + 1j * (omega_scan * L_line - 1.0 / (omega_scan * C_comp))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax1.plot(f_scan, np.abs(Z_line), 'b-', linewidth=2)
ax1.set_ylabel('|Z| (pu)'); ax1.set_title('Frequency Scan: System Impedance')
ax1.grid(True, alpha=0.3)
f_res = 50 * np.sqrt(0.5)
ax1.axvline(x=f_res, color='r', linestyle='--', label=f'f_res={f_res:.1f} Hz')
ax1.legend()

ax2.plot(f_scan, np.angle(Z_line, deg=True), 'r-', linewidth=2)
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=f_res, color='r', linestyle='--')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('Phase (deg)')
ax2.set_title('Impedance Phase Angle')
ax2.grid(True, alpha=0.3); ax2.set_xlim(5, 55)

plt.tight_layout()
plt.savefig('examples/kundur/ssr_frequency_scan.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Resonance at f_res = {f_res:.1f} Hz (minimum impedance)')
print('At resonance: X_L = X_C, system presents minimum impedance')

---
## 15.5 SSR 对策

### 预防措施
1. **避免危险补偿度**: 使 $f_{er}$ 不接近轴系扭振互补频率
2. **旁路滤波器** (Blocking Filter): 在升压变中性点加装阻波器
3. **SSR 阻尼控制器**: 通过励磁系统或 FACTS 设备提供次同步频率阻尼
4. **晶闸管控制串联电容器** (TCSC): 灵活调节等效补偿度

### 检测方法
- 扭振监测 (Torsional Vibration Monitoring)
- 次同步频率电流/电压检测
- 频率扫描和特征值分析


---
## 本章小结

1. **SSR 本质**: 串联补偿电容与系统电感形成电气谐振，
   与发电机轴系扭振耦合导致不稳定的次同步振荡
2. **补偿度** 决定了电气谐振频率：$f_{er} = f_0 \\sqrt{K}$
3. **感应发电机效应**: 次同步频率下发电机呈负电阻
4. **扭振相互作用**: 电气谐振频率接近轴系扭振互补频率时最危险
5. **频率扫描** 是评估 SSR 风险的基本工具

> Kundur 参考 (Ch.15, Sec. 15.2-15.4):
> - IEEE SSR Task Force: First and Second Benchmark Models
> - 典型扭振频率: 15-45 Hz (大型汽轮发电机)
> - 补偿度 30-70% 对应电气谐振 27-42 Hz
